# nano Beatrix v1.1 — the self-written frames prototype: the carry frame

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/AbstractEyes/geolip-nano-beatrix-trainer/blob/main/colab/nano_v1_1_carry.ipynb)

The frames continuation showed (2026-10-17) that a v1.1 craft reads a frame within about a thousand bytes of its question (copy 0.93 after 1.0B framed bytes) and cannot reach a record planted 2,048 bytes back or more. This notebook trains the first form of a craft that decides where a frame goes: the CARRY FRAME. A carry row holds one small lookup frame early (32 or 64 bytes, one or two records), then plain text with carries on the 64-byte grid every 384 to 640 bytes, each a copy of the early frame, and the question in the last 512 bytes asking any early record. The craft learns to write the carries; the gauge then reads it WITHOUT them (reading with notes): at every grid point its greedy next byte may open a carry, which it writes itself and which enters the row; the value is scored at the far question on its own row.

| craft (one a body) | starts from | its phase (1.0 billion bytes, 3,815 steps) |
|---|---|---|
| nano-<body>-v1.1-cf | the v1.1 craft's final state (step 61,422), the same state as the -fr and -ct arms | three rows in four carry rows, the rest plain fineweb-edu text (the same text row for row as -fr and -ct) |

for the four bodies nano-pollux-dual-canon, nano-pollux-canon, nano-pollux-dual and nano-castor. The replication (the frames arm, -fr) and the control (-ct) are the frames notebook's crafts, already on the Hub for the bodies it finished.

The start is exact (the fp32 weights and both optimizers' states from resume/boundary_anneal_mix_step61422.pt, the same recipe). The reads: the before row, the bed's frame gauges at 62,000, 64,000 and the end (comparable with -fr), the carry gauge at the same points, the full eval every 2,000 steps. C4 tabulates before, after frames, after carry, after control, and the carry gauge on the -cf craft beside the same read on the -fr craft (which never saw a carry and should write none).

Length: about 25 minutes a body on the card the frames train ran on, plus the before reads and the carry gauge (a few minutes a read). Needs: the v1.1 finals in the repo (C2 checks), and HF_TOKEN in this notebook's secrets with write access to AbstractPhil/alephllm-nano-castor-pollux.

In [ ]:
# C1 — pinned installs + THE DECISION BLOCK. RESTART RUNTIME if the version line below changes.
import os, sys
os.environ['HF_HUB_DISABLE_PROGRESS_BARS'] = '1'   # a progress bar is not allowed to cost a session
%pip install -q "geolip-nano-beatrix-trainer[corpus] @ git+https://github.com/AbstractEyes/geolip-nano-beatrix-trainer@v0.1.40"
import torch
import geolip.nano_beatrix as N
import geolip.alephllm as A
from geolip.nano_beatrix import nano_v1 as V1
print('geolip.nano_beatrix', N.__version__, '| geolip.alephllm', A.__version__, '| torch', torch.__version__,
      '| card', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
assert tuple(int(x) for x in N.__version__.split('.')) >= (0, 1, 40), (
    'RESTART REQUIRED: the runtime still holds an older package — Runtime > Restart, then rerun from C1')
assert torch.cuda.is_available(), 'no card: Runtime > Change runtime type > GPU'

def _token():
    try:
        from google.colab import userdata
        return userdata.get('HF_TOKEN')
    except Exception:
        return os.environ.get('HF_TOKEN')
HF_TOKEN = _token()                                   # the value is never printed
assert HF_TOKEN, 'HF_TOKEN missing: the start states are read from the repo and the continuations ship to it'
os.environ['HF_TOKEN'] = HF_TOKEN

# ============ THE DECISION BLOCK — every setting from the record, with its source beside it ============
OUT = '/content/nano_v1_1_carry'                     # the session folder: the corpus, the runs, the bed's gauge folder
REPO = V1.REPO                                        # AbstractPhil/alephllm-nano-castor-pollux: the v1.1 folders read, the new ones written
NAMES = [n for n in V1.CRAFTS_CONT if n.endswith('-cf')]   # the carry arm of each body (the -fr and -ct arms are the frames notebook's)
MB, GA = 16, 4                                        # the 2s shape, the fastest on the v1.1 benchmark (262,144 bytes a step)
GAUGE_EVERY = 2000                                    # the bed's gauges at 62,000 and 64,000 (every full eval) and at the end
CORPUS_BYTES = 1.0e9                                  # the frames corpus (fineweb-edu, the held-out head skipped), built once in C2
MAX_HOURS = None                                      # this session's budget in hours (None: until every craft ends or the cell is stopped)
print('decision block:', len(NAMES), 'crafts:', ', '.join(NAMES), '| the phase', V1.CONT_TOKENS / 1e9, 'B bytes |', MB, 'x', GA,
      '| gauges every', GAUGE_EVERY, 'steps | corpus', CORPUS_BYTES / 1e9, 'GB | ships to', REPO)

In [ ]:
# C2 (P) — PREFLIGHT: every v1.1 craft's final state in the repo, the held-out head, the frames corpus (built once; about a minute or two
# a gigabyte from the stream), the rows' composition pace against the card's, the push probe. Nothing trains here.
import json, time
from huggingface_hub import HfApi
from geolip.nano_beatrix.bed_corpus import write_heldout, build as build_corpus
from geolip.nano_beatrix import frames_data as FD
api = HfApi(token=HF_TOKEN)
files = set(api.list_repo_files(REPO))
missing = []
for n in NAMES:
    spec = V1.cont_init(n, MB * GA * V1.CTX)
    path = f"{spec['prefix']}/{spec['payload']}"
    if path not in files:
        missing.append(path)
assert not missing, f'not every v1.1 craft has finished: missing {sorted(set(missing))}'
print('start states: every v1.1 craft has its final state at step', spec['step'])
held = write_heldout(f'{OUT}/bed')
folder = f'{OUT}/frames_corpus'
if not os.path.exists(f'{folder}/bed_corpus.bin'):
    build_corpus(folder, int(CORPUS_BYTES))
packed = FD.load_corpus(folder)
n, t0 = 0, time.time()
while time.time() - t0 < 10:
    FD.row_bytes(packed, 1337, n, FD.P_FRAMED, FD.CARRY, carry=True)
    n += 1
rate = n / (time.time() - t0)
need = 879_000 / FD.ROW_BYTES                         # the fastest craft's rows a second (Castor, 879k bytes a second)
print(f'frames corpus {len(packed) / 1e9:.2f} GB; the frames rows compose at {rate:,.0f} a second against {need:.0f} needed'
      + ('' if rate > 1.5 * need else ' — TOO SLOW: the card will wait on the data'))

PUSH_OK = False
path = 'preflight/push_probe_v1_1_carry.json'        # its own probe file
body = json.dumps({'utc': time.strftime('%Y-%m-%dT%H:%M:%SZ', time.gmtime()), 'card': torch.cuda.get_device_name(0),
                   'compose_rows_s': rate, 'crafts': NAMES}).encode()
api.upload_file(path_or_fileobj=body, path_in_repo=path, repo_id=REPO, commit_message='nano v1.1 carry: push probe')
PUSH_OK = path in set(api.list_repo_files(REPO))
print('push probe', 'READ BACK OK' if PUSH_OK else 'NOT VISIBLE after upload', path)

In [ ]:
# C3 (T) — THE CONTINUATIONS: each craft starts from its v1.1 craft's final state (its first row: the before, read by the gauges), then
# runs its phase; the trainer's lines and the bed's gauge lines ([v1]) print as they come. Rerun the cell to continue (a new session
# too). The cell's stop button saves the running craft's state before it stops.
assert PUSH_OK, 'the push probe did not read back: nothing would ship'
HIST = V1.run(OUT, HF_TOKEN, NAMES, micro_batch=MB, grad_accum=GA, repo=REPO, gauge_every=GAUGE_EVERY, max_hours=MAX_HOURS,
              frames_corpus=f'{OUT}/frames_corpus')
from IPython.display import Markdown, display
display(Markdown(V1.cont_table(HIST)))

In [ ]:
# C4 — BEFORE AND AFTER, AND READING WITH NOTES: the boundary rows of every continuation of these bodies from the repo (the -fr and -ct
# arms from the frames notebook where they exist), the four-column table, then the carry gauge on the -cf craft's end beside the same
# read on the -fr craft's end (gauge_stored pulls its final and runs the gauge here: a few minutes a body). Run any time; it trains nothing.
import json
from IPython.display import Markdown, display
from huggingface_hub import snapshot_download
BODIES = [n[:-3] for n in NAMES]
ALL_ARMS = [f'{b}-{a}' for b in BODIES for a in ('fr', 'ct', 'cf')]
hub = snapshot_download(REPO, token=HF_TOKEN, local_dir=f'{OUT}/hub',
                        allow_patterns=[f'{n}/runs/*' for n in ALL_ARMS] + [f'{n}/v1_boundaries.json' for n in ALL_ARMS])
hist = {}
for n in ALL_ARMS:
    p = f'{hub}/{n}/v1_boundaries.json'
    hist[n] = json.load(open(p, encoding='utf-8')) if os.path.exists(p) else []
display(Markdown(V1.cont_table(hist)))
END_STEP = V1.plan_steps(NAMES[0], MB * GA * V1.CTX)      # 65,237, the whole plan's steps (0.1.40; a floor division read 65,236 before)
rows = ['| body | craft | whole value at the far question, own carries | first byte | value bits | carries a row | rows with a carry | exact copies | open probability at the grid | distance to the latest definition |',
        '|---|---|---|---|---|---|---|---|---|---|']
for b in BODIES:
    for arm in ('cf', 'fr'):
        n = f'{b}-{arm}'
        if not any(e.get('phase') == V1.CONT_PHASE[arm][0] for e in hist.get(n, [])):
            rows.append(f'| {b} | {arm} | not finished | | | | | | | |')
            continue
        c = V1.gauge_stored(n, END_STEP, OUT, repo=REPO, carry=True, token=HF_TOKEN).get('carry') or {}
        rows.append(f"| {b} | {arm} | {c.get('full_value', float('nan')):.3f} | {c.get('first_byte', float('nan')):.3f} | {c.get('value_bits', float('nan')):.1f} | "
                    f"{c.get('carries_per_row', float('nan')):.2f} | {c.get('rows_with_carries', float('nan')):.2f} | {c.get('carry_exact', float('nan')):.2f} | "
                    f"{c.get('open_prob', float('nan')):.3f} | {c.get('dist_latest', float('nan')):.0f} |")
display(Markdown('\n'.join(rows)))
display(Markdown(V1.table(hist)))
%load_ext tensorboard
%tensorboard --logdir {hub}


### Changing the run

Edit the decision block in C1, rerun C1 (the install is cached), then C2 and C3. A craft already in the repo resumes; it never restarts.

| to | the edit in C1 |
|---|---|
| one body first | `NAMES = ['nano-castor-v1.1-fr', 'nano-castor-v1.1-ct']` (the others wait) |
| a session budget | `MAX_HOURS = 2` |
| two cards | the first card `NAMES = NAMES[:4]`, the second `NAMES = NAMES[4:]` (two bodies each, every pair whole) |

The session folder is the Colab disk: the repo holds the records.